# 03 PySpark ML Fraud Modeling

This notebook builds supervised machine learning models in PySpark to predict fraudulent credit card transactions.

The modeling workflow includes:
- loading the transaction data with Spark
- applying shared cleaning and feature engineering
- training a Logistic Regression baseline
- comparing performance against a Random Forest model
- evaluating tradeoffs between predictive performance and operational speed

In [1]:
#Imports
from pathlib import Path

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.window import Window


from project.config import load_config
from project.spark_session import create_spark
from project.schemas import TRANSACTION_SCHEMA
from project.transform import prepare_transactions
from project.fraud_modeling import split_by_label, undersample_training, oversample_training, hybrid_sample_training, fit_and_score_model, summarize_binary_predictions, build_comparison_df, show_summary, run_experiment

## Load configuration and start Spark

We load the project configuration and start a local Spark session.  
This keeps the notebook aligned with the same reusable project structure used in the EDA and streaming notebooks.

In [2]:
repo_root = Path.cwd().parent
cfg = load_config(repo_root / "configs" / "local.yaml")

raw_csv = repo_root / cfg.paths.raw

spark = create_spark(cfg)
spark.sparkContext.setLogLevel("WARN")

print("Spark session created.")
print(f"Raw data path: {raw_csv}")

Spark session created.
Raw data path: /home/jovyan/work/data/raw/credit_card_transactions.csv


## Read the dataset with Spark

We load the full transaction dataset using the shared transaction schema.  
The same schema is reused across the project for consistency.

In [3]:
df = (spark.read
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("dateFormat", "yyyy-MM-dd")
         .schema(TRANSACTION_SCHEMA)
         .csv(str(raw_csv))
)

In [4]:
df.printSchema()
print(f"Row count: {df.count():,}")

root
 |-- Unnamed: 0: long (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merch_zipcode: string (nullable = true)

Row count: 1,296,675


## Apply reusable data transformations

We apply the shared transformation pipeline from `transform.py` to clean and enrich the dataset.

This step adds modeling-friendly features such as:
- transaction hour
- day of week
- customer age
- log-transformed transaction amount

In [5]:
df = prepare_transactions(df)

In [6]:
df.select(
    "amt",
    "category",
    "gender",
    "state",
    "event_hour",
    "event_dayofweek",
    "age",
    "is_fraud"
).show(10, truncate=False)

+------+-------------+------+-----+----------+---------------+---+--------+
|amt   |category     |gender|state|event_hour|event_dayofweek|age|is_fraud|
+------+-------------+------+-----+----------+---------------+---+--------+
|4.97  |misc_net     |F     |NC   |0         |3              |38 |0       |
|107.23|grocery_pos  |F     |WA   |0         |3              |47 |0       |
|220.11|entertainment|M     |ID   |0         |3              |64 |0       |
|45.0  |gas_transport|M     |MT   |0         |3              |59 |0       |
|41.96 |misc_pos     |M     |VA   |0         |3              |40 |0       |
|94.63 |gas_transport|F     |PA   |0         |3              |64 |0       |
|44.54 |grocery_net  |F     |KS   |0         |3              |32 |0       |
|71.65 |gas_transport|M     |VA   |0         |3              |78 |0       |
|4.27  |misc_pos     |F     |PA   |0         |3              |85 |0       |
|198.39|grocery_pos  |F     |TN   |0         |3              |52 |0       |
+------+----

## Select modeling features

For the first-pass fraud model, we use a focused set of numeric and categorical features that are likely to be predictive while remaining efficient and interpretable.

We avoid identifier columns such as account numbers and transaction IDs in the initial model.

In [7]:
numeric_features = [
    "amt",
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [8]:
model_cols = numeric_features + categorical_features + [label_col]

model_df = df.select(*model_cols).dropna()

print(f"Modeling row count after dropna: {model_df.count():,}")
model_df.groupBy("is_fraud").count().show()

Modeling row count after dropna: 1,296,675
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       1|   7506|
|       0|1289169|
+--------+-------+



## Split the data into training and test sets

We create a train/test split so that model performance can be evaluated on unseen transactions.

In [9]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=5420)

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

Train rows: 1,036,611
Test rows: 260,064


## Build the PySpark feature pipeline

We encode categorical variables, assemble numeric and encoded categorical features into a single feature vector, and pass that vector to the classifier.

In [10]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features"
)

## Train a Logistic Regression baseline

Logistic Regression provides a strong baseline because it is fast, scalable, and relatively interpretable for binary classification problems such as fraud detection.

In [11]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

In [12]:
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

In [13]:
lr_predictions.select(
    "is_fraud",
    "prediction",
    "probability"
).show(10, truncate=False)

+--------+----------+------------------------------------------+
|is_fraud|prediction|probability                               |
+--------+----------+------------------------------------------+
|0       |0.0       |[0.9991644311406339,8.35568859366087E-4]  |
|0       |0.0       |[0.9987990673999426,0.0012009326000573806]|
|0       |0.0       |[0.9965662824442785,0.003433717555721527] |
|0       |0.0       |[0.999554536814501,4.4546318549898434E-4] |
|0       |0.0       |[0.9945435793357229,0.005456420664277095] |
|0       |0.0       |[0.9925157279641066,0.007484272035893413] |
|0       |0.0       |[0.9937814644207845,0.006218535579215478] |
|0       |0.0       |[0.9985417508561539,0.0014582491438460687]|
|0       |0.0       |[0.9958755001952793,0.004124499804720738] |
|0       |0.0       |[0.9977058093848294,0.002294190615170555] |
+--------+----------+------------------------------------------+
only showing top 10 rows


## Evaluate Logistic Regression

Because fraud detection is an imbalanced classification problem, accuracy alone is not sufficient.  
We evaluate the model using AUC, precision, recall, and F1.

In [14]:
binary_eval = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="f1"
)

precision_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_eval = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="weightedRecall"
)

In [15]:
lr_auc = binary_eval.evaluate(lr_predictions)
lr_f1 = f1_eval.evaluate(lr_predictions)
lr_precision = precision_eval.evaluate(lr_predictions)
lr_recall = recall_eval.evaluate(lr_predictions)

print("Logistic Regression Metrics")
print(f"AUC:       {lr_auc:.4f}")
print(f"F1:        {lr_f1:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall:    {lr_recall:.4f}")

Logistic Regression Metrics
AUC:       0.8268
F1:        0.9911
Precision: 0.9893
Recall:    0.9937


In [16]:
lr_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction").show()

+--------+----------+------+
|is_fraud|prediction| count|
+--------+----------+------+
|       0|       0.0|258388|
|       0|       1.0|   124|
|       1|       0.0|  1524|
|       1|       1.0|    28|
+--------+----------+------+



## Train a Random Forest comparison model

Random Forest can capture nonlinear patterns and interactions that Logistic Regression may miss.  
We compare its predictive performance against the faster, simpler baseline.

In [17]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=10,
    seed=5420
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

In [18]:
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

In [19]:
rf_auc = binary_eval.evaluate(rf_predictions)
rf_f1 = f1_eval.evaluate(rf_predictions)
rf_precision = precision_eval.evaluate(rf_predictions)
rf_recall = recall_eval.evaluate(rf_predictions)

print("Random Forest Metrics")
print(f"AUC:       {rf_auc:.4f}")
print(f"F1:        {rf_f1:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")

Random Forest Metrics
AUC:       0.9852
F1:        0.9950
Precision: 0.9960
Recall:    0.9960


In [20]:
rf_predictions.groupBy("is_fraud", "prediction").count().orderBy("is_fraud", "prediction").show()

+--------+----------+------+
|is_fraud|prediction| count|
+--------+----------+------+
|       0|       0.0|258512|
|       1|       0.0|  1043|
|       1|       1.0|   509|
+--------+----------+------+



## Compare model performance

We compare the two models to assess the tradeoff between:
- predictive power
- model complexity
- operational suitability for near-real-time fraud scoring

In [21]:
comparison_rows = [
    ("Logistic Regression", round(lr_auc, 6), round(lr_f1, 6), round(lr_precision, 6), round(lr_recall, 6)),
    ("Random Forest", round(rf_auc, 6), round(rf_f1, 6), round(rf_precision, 6), round(rf_recall, 6)),
]

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["model", "auc", "f1", "precision", "recall"]
)

comparison_df.show()

+-------------------+--------+--------+---------+--------+
|              model|     auc|      f1|precision|  recall|
+-------------------+--------+--------+---------+--------+
|Logistic Regression|0.826809|0.991068| 0.989303|0.993663|
|      Random Forest| 0.98518|0.994979| 0.996006|0.995989|
+-------------------+--------+--------+---------+--------+



## Model Performance Observation

At first glance, both Logistic Regression and Random Forest models appear to perform extremely well, with high AUC, F1, precision, and recall scores.

However, these metrics are misleading due to the extreme class imbalance in the dataset.

Fraudulent transactions represent only a small fraction of the data. As a result, the models can achieve high overall performance by simply predicting the majority class (non-fraud) for most observations.

This behavior is confirmed by examining prediction outputs and confusion matrices, where the model rarely identifies fraudulent transactions correctly.

Key issue:
- The reported F1, precision, and recall are **weighted metrics**, which are dominated by the majority class (non-fraud)
- As a result, strong performance on non-fraud inflates the overall metrics
- The model still performs poorly at detecting fraud, which is the primary objective

This highlights a fundamental challenge in fraud detection:
- Standard evaluation metrics can be misleading under class imbalance
- Models must be specifically optimized to detect minority class behavior

To address this, the next steps include:
- applying class weighting to emphasize fraud cases
- engineering features that better capture anomalous behavior
- re-evaluating model performance with improved techniques

In [22]:
window = Window.partitionBy()

print("Logistic Regression – Fraud-only performance")

lr_fraud = lr_predictions.filter(F.col("is_fraud") == 1)

lr_fraud_summary = (
    lr_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

lr_fraud_summary.show()

Logistic Regression – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1524|       1552|      98.2|
|       1.0|   28|       1552|       1.8|
+----------+-----+-----------+----------+



In [23]:
print("Random Forest – Fraud-only performance")

rf_fraud = rf_predictions.filter(F.col("is_fraud") == 1)

rf_fraud_summary = (
    rf_fraud
    .groupBy("prediction")
    .count()
    .withColumn("total_fraud", F.sum("count").over(window))
    .withColumn("percentage", F.col("count") / F.col("total_fraud"))
    .withColumn("percentage", F.round(F.col("percentage") * 100, 2))
    .orderBy("prediction")
)

rf_fraud_summary.show()

Random Forest – Fraud-only performance
+----------+-----+-----------+----------+
|prediction|count|total_fraud|percentage|
+----------+-----+-----------+----------+
|       0.0| 1043|       1552|      67.2|
|       1.0|  509|       1552|      32.8|
+----------+-----+-----------+----------+



### Fraud Detection Performance

The fraud-only evaluation highlights a critical issue with both models.

- Logistic Regression correctly identifies only **~2%** of fraud cases
- Random Forest improves performance but still captures only **~33%** of fraud cases

This means that the majority of fraudulent transactions are being classified as non-fraud.

Despite strong overall evaluation metrics, both models perform poorly on the actual objective of fraud detection due to class imbalance. The models are biased toward predicting the majority class (non-fraud), which dominates the dataset.

### Next Steps: Improving Fraud Detection

To address these limitations, we will improve the modeling approach using the following strategies:

1. **Class Imbalance Handling**
   - Oversampling/Undersampling
   - Apply class weighting to penalize misclassification of fraud cases more heavily

2. **Feature Engineering**
   - Introduce features that better capture anomalous behavior, such as:
     - transactions occurring at unusual times (e.g., late night)
     - abnormal transaction amounts
     - geographic distance between customer and merchant

3. **Model Re-evaluation**
   - Retrain models using the improved features and weighting
   - Compare fraud detection performance before and after improvements

These steps aim to increase the model’s ability to detect fraudulent transactions while maintaining scalability for real-world applications.

### Modeling note

At this stage, imbalance experiments are being tested with Logistic Regression only. This allows faster iteration and clearer comparison across resampling strategies.

Because Random Forest is more computationally expensive, it will be reintroduced later once the most effective class-balancing approach has been identified.

## Imbalance Handling

The baseline models performed poorly for fraud detection because the dataset is highly imbalanced. Most transactions are non-fraud, so the models tend to favor the majority class.

To address this, the following resampling strategies are tested:

1. **Undersampling**  
   Reduce the number of non-fraud transactions in the training set.

2. **Oversampling**  
   Duplicate fraud transactions in the training set.

3. **Hybrid sampling**  
   Combine limited oversampling of fraud with undersampling of non-fraud.

Important:
- Resampling is applied **only to the training data**
- The **test set remains untouched**
- This ensures model evaluation reflects performance on real-world class imbalance rather than an artificially balanced test set

In [24]:
fraud_train_df, nonfraud_train_df = split_by_label(train_df, label_col="is_fraud")

fraud_train_count = fraud_train_df.count()
nonfraud_train_count = nonfraud_train_df.count()

print(f"Fraud rows in training set: {fraud_train_count:,}\nNon-fraud rows in training set: {nonfraud_train_count:,}")

Fraud rows in training set: 5,954
Non-fraud rows in training set: 1,030,657


### Why resample only the training set?

If oversampling or undersampling is applied before the train/test split, duplicated or reduced examples may distort evaluation results.

By keeping the test set unchanged, all models are evaluated on the same original class distribution. This provides a more realistic comparison of fraud detection performance.

### Undersampling

Undersampling reduces the majority class (non-fraud) to better match the minority class (fraud).

This helps the model focus on fraud patterns, but it may discard useful information from valid transactions.

In [25]:
under_train_balanced_df = undersample_training(
    fraud_train_df,
    nonfraud_train_df,
    label_col="is_fraud",
    seed=5420
)

print(f"Undersampled training rows: {under_train_balanced_df.count():,}")
under_train_balanced_df.groupBy("is_fraud").count().show()

under_lr_predictions, under_lr_summary = run_experiment(
    lr_pipeline,
    under_train_balanced_df,
    test_df,
    model_name="Undersampled Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

show_summary(spark, under_lr_summary)

Undersampled training rows: 11,947
+--------+-----+
|is_fraud|count|
+--------+-----+
|       1| 5954|
|       0| 5993|
+--------+-----+

+--------------------------------+------+-----+---+----+---------------+------------+--------+
|model                           |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|
+--------------------------------+------+-----+---+----+---------------+------------+--------+
|Undersampled Logistic Regression|226794|31718|373|1179|0.0358         |0.7597      |0.0684  |
+--------------------------------+------+-----+---+----+---------------+------------+--------+



#### Undersampling Results

With undersampling, our Logistic Regression model, undergoing no parameter changes, was able to increase its accuracy to 76%. A substantial increase from our initial outcome of ~1%.

### Oversampling

Oversampling increases the number of fraud transactions by duplicating minority-class examples.

This preserves all non-fraud data, but repeated fraud examples may increase the risk of overfitting.

In [26]:
over_train_balanced_df = oversample_training(
    fraud_train_df,
    nonfraud_train_df,
    label_col="is_fraud",
    seed=5420
)

print(f"Oversampled training rows: {over_train_balanced_df.count():,}")
over_train_balanced_df.groupBy("is_fraud").count().show()

over_lr_predictions, over_lr_summary = run_experiment(
    lr_pipeline,
    over_train_balanced_df,
    test_df,
    model_name="Oversampled Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [over_lr_summary]).show(truncate=False)

Oversampled training rows: 2,061,436
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       0|1030657|
|       1|1030779|
+--------+-------+

+-------------------------------+------+-----+---+----+---------------+------------+--------+
|model                          |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|
+-------------------------------+------+-----+---+----+---------------+------------+--------+
|Oversampled Logistic Regression|227607|30905|371|1181|0.0368         |0.761       |0.0702  |
+-------------------------------+------+-----+---+----+---------------+------------+--------+



#### Oversampling Results

With oversampling, our Logistic Regression model increased fraud recall to 76%, a substantial improvement over the baseline fraud recall of about 1%.

### Imbalance Summary

Based on the outcomes, undersampling and oversampling led to roughly same outcome of ~75% fraud recall on fraudulent charges. Undersampling, being only ~15,000 rows of data, was able to produce an outcome much quicker and with similiar results with less data. With oversampling, there is much more data to evaluate, but with the fraudulent data needing to be duplicated over 150 times, this very likely would lead to overfitting once we tweek our hyper parameters on our models since it would recognize the same datapoints reproduced that many times. 

### Imbalance Conclusion

The initial resampling experiments suggest that a hybrid approach may provide a better balance between information retention and class balance than pure undersampling or pure oversampling alone.

Undersampling performed well with much lower computational cost, while oversampling preserved majority-class data but required substantial duplication of fraud observations. Hybrid sampling is intended to balance these tradeoffs and should be evaluated using fraud recall, fraud precision, and fraud F1 on the unchanged test set.

### Hybrid Sampling

Hybrid sampling combines limited oversampling of fraud with undersampling of non-fraud.

This approach aims to:
- avoid discarding too much majority-class data
- avoid excessive duplication of minority-class data
- create a more balanced training set using a middle-ground strategy

In [27]:
hybrid_train_df = hybrid_sample_training(
    fraud_train_df,
    nonfraud_train_df,
    fraud_multiplier=5.0,
    seed=5420
)

print(f"Hybrid training rows: {hybrid_train_df.count():,}")
hybrid_train_df.groupBy("is_fraud").count().show()

hybrid_lr_predictions, hybrid_lr_summary = run_experiment(
    lr_pipeline,
    hybrid_train_df,
    test_df,
    model_name="Hybrid Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

build_comparison_df(spark, [hybrid_lr_summary]).show(truncate=False)

Hybrid training rows: 59,540
+--------+-----+
|is_fraud|count|
+--------+-----+
|       1|29770|
|       0|29770|
+--------+-----+

+--------------------------+------+-----+---+----+---------------+------------+--------+
|model                     |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|
+--------------------------+------+-----+---+----+---------------+------------+--------+
|Hybrid Logistic Regression|228549|29963|370|1182|0.038          |0.7616      |0.0723  |
+--------------------------+------+-----+---+----+---------------+------------+--------+



#### Hybrid Sampling Results

Hybrid sampling combines moderate fraud oversampling with non-fraud undersampling to create a more balanced training set without fully discarding majority-class information or excessively duplicating minority-class examples.

This method should be compared against the undersampling and oversampling results using fraud recall, fraud precision, and fraud F1 on the unchanged test set.

In [28]:
baseline_lr_summary = run_experiment(
    lr_pipeline,
    train_df,
    test_df,
    model_name="Baseline Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)[1]

comparison_df = build_comparison_df(
    spark,
    [
        baseline_lr_summary,
        under_lr_summary,
        over_lr_summary,
        hybrid_lr_summary,
    ]
)

comparison_df.show(truncate=False)

+--------------------------------+------+-----+----+----+---------------+------------+--------+
|model                           |tn    |fp   |fn  |tp  |fraud_precision|fraud_recall|fraud_f1|
+--------------------------------+------+-----+----+----+---------------+------------+--------+
|Baseline Logistic Regression    |258388|124  |1524|28  |0.1842         |0.018       |0.0329  |
|Undersampled Logistic Regression|226794|31718|373 |1179|0.0358         |0.7597      |0.0684  |
|Oversampled Logistic Regression |227607|30905|371 |1181|0.0368         |0.761       |0.0702  |
|Hybrid Logistic Regression      |228549|29963|370 |1182|0.038          |0.7616      |0.0723  |
+--------------------------------+------+-----+----+----+---------------+------------+--------+



Conclusion: With all methods performing roughy the same. It becomes a choice between the Hybrid method and the Undersample method.

Oversampling takes too long to run and operationally would be far too expensive. Hybrid is likely better since it allows for an "artificially" larger dataset with duplicated some fraud data without going overboard and pulling in additional data from the true legit credit card transactions.

Next Steps:

## Class Weights

Another method to address the imbalance, class weights will be tested. Rather than changing the class distribution through undersampling or oversampling, weighting changes the training loss so that errors on fraudulent transactions carry a larger penalty than errors on non-fraudulent ones.

In practice, this means:

- the fraud class receives a higher weight/penalty
- the non-fraud class receives a lower weight/penalty
- the model is encouraged to pay more attention to the minority class

This approach helps improve fraud detection while preserving the full training dataset. Compared with resampling, class weighting avoids removing legitimate transactions or duplicating fraudulent ones, making it a useful alternative for handling imbalance in an interpretable way.

In [29]:
# Calculating weights (using the train df)
total_count = train_df.count()

fraud_count = train_df.filter(F.col("is_fraud") == 1).count()
nonfraud_count = train_df.filter(F.col("is_fraud") == 0).count()

num_classes = 2

weight_0 = total_count / (num_classes * nonfraud_count) #majority
weight_1 = total_count / (num_classes * fraud_count) #minority

In [30]:
print(f"With the weights now in place;\n\tNon-fraud data is weighted as {round(weight_0,4)}\n\tFraud data is weighted as {round(weight_1,4)}\nWith these weights in place, the contributions are now equalized without altering our actual data\n\tNon-fraud contribution: {nonfraud_count * weight_0}\n\tFraud contribution: {fraud_count * weight_1}" )

With the weights now in place;
	Non-fraud data is weighted as 0.5029
	Fraud data is weighted as 87.0516
With these weights in place, the contributions are now equalized without altering our actual data
	Non-fraud contribution: 518305.5
	Fraud contribution: 518305.5


In [31]:
train_weighted = train_df.withColumn(
    "class_weight",
    F.when(F.col("is_fraud") == 1, weight_1).otherwise(weight_0)
)

test_weighted = test_df.withColumn("class_weight", F.lit(1.0))

In [32]:
# Slight alteration to our logistic regression model so it can use weights
lr_w = LogisticRegression(
    featuresCol="features",
    weightCol="class_weight", # This is the only change, all other parameters preserved
    labelCol=label_col,
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

lr_w_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr_w])

In [33]:
weighted_preds, weighted_summary = run_experiment(
    lr_w_pipeline,
    train_weighted,
    test_weighted,
    model_name="Weighted Logistic Regression",
    label_col="is_fraud",
    pred_col="prediction"
)

In [34]:
build_comparison_df(spark, [weighted_summary]).show(truncate=False)

+----------------------------+------+-----+---+----+---------------+------------+--------+
|model                       |tn    |fp   |fn |tp  |fraud_precision|fraud_recall|fraud_f1|
+----------------------------+------+-----+---+----+---------------+------------+--------+
|Weighted Logistic Regression|227841|30671|371|1181|0.0371         |0.761       |0.0707  |
+----------------------------+------+-----+---+----+---------------+------------+--------+



### Weighted Results

The results are very similar to our earlier methods of bringing out dataset in to balance. We will compare all methods outcomes below and decide on how to proceed.

## Imbalance Correction Results

In [35]:
comparison_df = build_comparison_df(
    spark,
    [
        baseline_lr_summary,
        under_lr_summary,
        over_lr_summary,
        hybrid_lr_summary,
        weighted_summary
    ]
)

comparison_df.show(truncate=False)

+--------------------------------+------+-----+----+----+---------------+------------+--------+
|model                           |tn    |fp   |fn  |tp  |fraud_precision|fraud_recall|fraud_f1|
+--------------------------------+------+-----+----+----+---------------+------------+--------+
|Baseline Logistic Regression    |258388|124  |1524|28  |0.1842         |0.018       |0.0329  |
|Undersampled Logistic Regression|226794|31718|373 |1179|0.0358         |0.7597      |0.0684  |
|Oversampled Logistic Regression |227607|30905|371 |1181|0.0368         |0.761       |0.0702  |
|Hybrid Logistic Regression      |228549|29963|370 |1182|0.038          |0.7616      |0.0723  |
|Weighted Logistic Regression    |227841|30671|371 |1181|0.0371         |0.761       |0.0707  |
+--------------------------------+------+-----+----+----+---------------+------------+--------+



Given that all imbalance-handling methods produced similar performance, the preferred approach is the one that minimizes computational cost while maintaining effectiveness. 

In this case, undersampling and weighted logistic regression provide comparable results to more computationally intensive methods, making them more practical choices for further modeling and potential real-world deployment.